# Sistema de control de peaje: modelos e inferencia en Google Colab

**IAM-811 · Laboratorio final de hito.** Todo en este único notebook: entorno, datasets de Roboflow, entrenamiento de **M2** (segmentación) y **M3** (keypoints de ruedas), validación de **M1** (placa + OCR) y el **endpoint de inferencia** con los 3 modelos que usa la web.

## Cómo usarlo

1. **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4).**
2. Completa la celda **1. Configuración** (clave y proyectos de Roboflow). La clave puede ir en **Secretos** (ícono de llave) con el nombre `ROBOFLOW_API_KEY`.
3. **Entorno de ejecución → Ejecutar todo.**
4. Al final, la celda **10** muestra el **link** y el **token**. En la web: **Estado de la IA → Conexión con Colab** → pégalos → **Guardar y probar**.
5. Deja Colab abierto con la celda **11** corriendo mientras se usa el sistema. **El link y el token cambian cada vez que se reinicia el notebook.**

Una sola sesión sirve para todo: el mismo endpoint hace la inferencia de imágenes **y** el agente conversacional (Qwen con Ollama).

## Qué hace cada sección

| Celda | Qué hace | Para el informe |
|---|---|---|
| 1 | Configuración | |
| 2 | Entorno con versiones fijas + diagnóstico (Python, torch, CUDA, GPU, ultralytics) | Bonus de entornos |
| 3 | Carpeta de trabajo en Google Drive (los pesos entrenados no se pierden al cerrar) | |
| 4 | Descarga los datasets de Roboflow y cuenta imágenes y clases | Datasets |
| 5 | Entrena **M2** (YOLO-seg: moto, auto, minibús, camión, bus) | mAP de M2 |
| 6 | Entrena **M3** (YOLO-pose: 1 keypoint en el centro de cada rueda) | mAP de M3 y error en el conteo de ejes |
| 7 | **M1**: detector de placas + OCR, y precisión del OCR | Precisión OCR |
| 8 | Carga los modelos para la inferencia y el LLM del agente (Qwen con Ollama) | |
| 9 | `analizar()`: los 3 modelos sobre una imagen + prueba | |
| 10 | Endpoint FastAPI (`/health`, `/inferir`, `/agente/chat`) + túnel de Cloudflare | |
| 11 | Mantiene la sesión activa y muestra la latencia | Latencia |

**Modelos que usa (ya entrenados, se descargan solos; no hace falta clave de Roboflow):**

| Modelo | Qué es | Para qué |
|---|---|---|
| M1 | YOLOv9-t 384 de placas + OCR CCT-XS v2 (proyecto `placas-web`) | Placa y texto |
| M2 | YOLO26n-seg entrenado en COCO (auto, moto, bus, camión) + "Van" de Open Images para minibús | Categoría (máscara por vehículo) y pasajeros |
| M3 | YOLOv8s entrenado en Open Images V7 (clases *Wheel* y *Tire*): centro de cada rueda = keypoint | Ejes |

Las celdas 4 a 6 (datasets y entrenamiento) quedan opcionales: con la configuración por defecto se saltan solas. Si algún día entrenan M2 y M3 propios, sus pesos en Drive se usan automáticamente.

**Privacidad (guía del lab):** las personas solo se **cuentan** (pasajeros). En la evidencia se cubren con su máscara en color sólido. No hay reconocimiento facial ni recortes de rostros.

In [ ]:
# @title 1. Configuración (edita solo esta celda)
import os, sys

EN_COLAB = "google.colab" in sys.modules

# Roboflow. La clave también puede ir en Secretos de Colab con el nombre ROBOFLOW_API_KEY.
ROBOFLOW_API_KEY = ""
# M2: segmentación de instancias. Clases: moto, auto, minibus, camion, bus (una máscara por vehículo).
M2_ROBOFLOW = {"workspace": "", "proyecto": "", "version": 1}
# M3: keypoints. Cada rueda visible es un objeto con 1 keypoint en su centro.
M3_ROBOFLOW = {"workspace": "", "proyecto": "", "version": 1}

# Decisión del equipo: se usan modelos YA ENTRENADOS (se descargan solos, no hace falta Roboflow).
# Pon True solo si algún día completan M2_ROBOFLOW / M3_ROBOFLOW y quieren entrenar los propios.
ENTRENAR_M2 = False
ENTRENAR_M3 = False
EPOCAS_M2 = 60
EPOCAS_M3 = 80
TAMANO = 640           # tamaño de entrada de los modelos

# Agente conversacional: LLM Qwen servido con Ollama en esta misma sesión (mismo link y token).
# "qwen2.5:3b" responde rápido en la T4 (~2 GB). Más nuevo y un poco más lento: "qwen3.5:4b".
# Con más calidad y más lento: "qwen2.5:7b".
USAR_AGENTE = True
AGENTE_MODELO = "qwen2.5:3b"

USAR_DRIVE = True      # guarda pesos, métricas y diagnóstico en Google Drive
INFERENCE_TOKEN = ""   # vacío = se genera uno nuevo (va en INFERENCE_TOKEN de Laravel)
PUERTO = 8001
USAR_TUNEL = EN_COLAB  # túnel público de Cloudflare (solo tiene sentido en Colab)

if EN_COLAB and not ROBOFLOW_API_KEY:
    try:
        from google.colab import userdata
        ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY") or ""
    except Exception:
        pass

print("En Colab:", EN_COLAB, "| Roboflow configurado:", bool(ROBOFLOW_API_KEY))

In [ ]:
# @title 2. Entorno: versiones fijas + diagnóstico (bonus de gestión de entornos)
import subprocess, platform, json, datetime

REQUIREMENTS = """\
# Servicio: inferencia del peaje en Google Colab (torch, CUDA y numpy son los que trae Colab)
ultralytics==8.4.218
fast-plate-ocr[onnx]==1.1.0
onnxruntime==1.23.2
roboflow==1.7.3
fastapi==0.162.5
uvicorn==0.54.0
python-multipart==0.0.32
requests==2.34.2
opencv-python-headless==5.1.0.95
"""
with open("requirements.txt", "w") as f:
    f.write(REQUIREMENTS)

if EN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

import torch, ultralytics, onnxruntime, cv2, numpy as np

DIAGNOSTICO = {
    "fecha": datetime.datetime.now().isoformat(timespec="seconds"),
    "entorno": "Google Colab" if EN_COLAB else platform.node(),
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda_disponible": torch.cuda.is_available(),
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "ultralytics": ultralytics.__version__,
    "onnxruntime": onnxruntime.__version__,
    "opencv": cv2.__version__,
    "numpy": np.__version__,
}
print(json.dumps(DIAGNOSTICO, indent=2, ensure_ascii=False))
if EN_COLAB and not DIAGNOSTICO["cuda_disponible"]:
    print("\nAVISO: no hay GPU. Entorno de ejecución → Cambiar tipo de entorno → GPU T4. Sin GPU todo es mucho más lento.")

In [ ]:
# @title 3. Carpeta de trabajo (Google Drive)
from pathlib import Path

if EN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = Path("/content/drive/MyDrive/peaje")
else:
    CARPETA = Path("peaje_trabajo").resolve()

MODELOS = CARPETA / "modelos"
INFORME = CARPETA / "informe"
for c in (MODELOS, INFORME, CARPETA / "datasets", CARPETA / "placas_validacion"):
    c.mkdir(parents=True, exist_ok=True)

M2_PESOS = MODELOS / "m2_vehiculos_seg.pt"
M3_PESOS = MODELOS / "m3_ruedas_pose.pt"

_ = (INFORME / "diagnostico_entorno.json").write_text(json.dumps(DIAGNOSTICO, indent=2, ensure_ascii=False))
_ = (INFORME / "requirements.txt").write_text(REQUIREMENTS)
print("Carpeta de trabajo:", CARPETA)

In [ ]:
# @title 4. Datasets de Roboflow (M2 y M3)
import yaml

def descargar_dataset(cfg, nombre):
    """Descarga el dataset en formato YOLO y devuelve la ruta a data.yaml (o None)."""
    if not (ROBOFLOW_API_KEY and cfg["workspace"] and cfg["proyecto"]):
        print(f"{nombre}: sin configurar en la celda 1, se omite.")
        return None
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    version = rf.workspace(cfg["workspace"]).project(cfg["proyecto"]).version(cfg["version"])
    ds = version.download("yolov8", location=str(CARPETA / "datasets" / nombre), overwrite=True)
    return Path(ds.location) / "data.yaml"

def resumen_dataset(data_yaml):
    datos = yaml.safe_load(Path(data_yaml).read_text())
    raiz = Path(data_yaml).parent
    imagenes = {}
    for split in ("train", "valid", "test"):
        carpeta = raiz / split / "images"
        imagenes[split] = len(list(carpeta.glob("*"))) if carpeta.exists() else 0
    return {"clases": datos.get("names"), "kpt_shape": datos.get("kpt_shape"), "imagenes": imagenes,
            "total": sum(imagenes.values())}

DATASETS = {}
M2_DATA = descargar_dataset(M2_ROBOFLOW, "m2_segmentacion")
M3_DATA = descargar_dataset(M3_ROBOFLOW, "m3_ruedas")
for nombre, data in (("M2", M2_DATA), ("M3", M3_DATA)):
    if data:
        DATASETS[nombre] = resumen_dataset(data)
        print(nombre, json.dumps(DATASETS[nombre], ensure_ascii=False))

# Las aumentaciones de Roboflow se ven en la página de la versión del dataset.
# YOLO además aplica las suyas al entrenar (mosaic, flips, HSV, escala); quedan en runs/*/args.yaml.
_ = (INFORME / "datasets.json").write_text(json.dumps(DATASETS, indent=2, ensure_ascii=False))

In [ ]:
# @title 5. Entrenar M2: segmentación de instancias (YOLO-seg)
import shutil
from ultralytics import YOLO

METRICAS = json.loads((INFORME / "metricas.json").read_text()) if (INFORME / "metricas.json").exists() else {}

if ENTRENAR_M2 and M2_DATA:
    modelo = YOLO("yolo26n-seg.pt")  # parte de COCO y se ajusta al dataset propio
    modelo.train(data=str(M2_DATA), epochs=EPOCAS_M2, imgsz=TAMANO, batch=16, patience=20,
                 project=str(CARPETA / "runs"), name="m2", exist_ok=True)
    shutil.copy(CARPETA / "runs" / "m2" / "weights" / "best.pt", M2_PESOS)

if M2_PESOS.exists() and M2_DATA:
    v = YOLO(str(M2_PESOS)).val(data=str(M2_DATA), imgsz=TAMANO, verbose=False)
    METRICAS["M2"] = {
        "caja_mAP50": round(float(v.box.map50), 4), "caja_mAP50_95": round(float(v.box.map), 4),
        "mascara_mAP50": round(float(v.seg.map50), 4), "mascara_mAP50_95": round(float(v.seg.map), 4),
    }
    print("M2:", METRICAS["M2"])
else:
    print("M2: sin entrenar (se usará el preentrenado de COCO en la inferencia).")
_ = _ = _ = (INFORME / "metricas.json").write_text(json.dumps(METRICAS, indent=2, ensure_ascii=False))

In [ ]:
# @title 6. Entrenar M3: keypoints de ruedas (YOLO-pose) + error en el conteo de ejes
if ENTRENAR_M3 and M3_DATA:
    modelo = YOLO("yolo26n-pose.pt")
    modelo.train(data=str(M3_DATA), epochs=EPOCAS_M3, imgsz=TAMANO, batch=16, patience=25,
                 project=str(CARPETA / "runs"), name="m3", exist_ok=True)
    shutil.copy(CARPETA / "runs" / "m3" / "weights" / "best.pt", M3_PESOS)

def error_conteo_ejes(pesos, data_yaml, conf=0.4):
    """Compara ruedas detectadas vs etiquetadas por imagen de validación (= ejes vistos de costado)."""
    raiz = Path(data_yaml).parent / "valid"
    modelo = YOLO(str(pesos))
    errores, exactos = [], 0
    for img in sorted((raiz / "images").glob("*")):
        etiqueta = raiz / "labels" / (img.stem + ".txt")
        reales = len([l for l in etiqueta.read_text().splitlines() if l.strip()]) if etiqueta.exists() else 0
        predichas = len(modelo.predict(str(img), conf=conf, verbose=False)[0].boxes)
        errores.append(abs(predichas - reales))
        exactos += predichas == reales
    n = max(1, len(errores))
    return {"imagenes": len(errores), "error_absoluto_medio": round(sum(errores) / n, 3),
            "conteo_exacto": round(exactos / n, 3)}

if M3_PESOS.exists() and M3_DATA:
    v = YOLO(str(M3_PESOS)).val(data=str(M3_DATA), imgsz=TAMANO, verbose=False)
    METRICAS["M3"] = {
        "caja_mAP50": round(float(v.box.map50), 4), "caja_mAP50_95": round(float(v.box.map), 4),
        "pose_mAP50": round(float(v.pose.map50), 4), "pose_mAP50_95": round(float(v.pose.map), 4),
        "conteo_ejes": error_conteo_ejes(M3_PESOS, M3_DATA),
    }
    print("M3:", METRICAS["M3"])
else:
    print("M3: sin entrenar (se usará YOLOv8s de Open Images: ruedas -> centro = keypoint).")
_ = _ = _ = (INFORME / "metricas.json").write_text(json.dumps(METRICAS, indent=2, ensure_ascii=False))

In [ ]:
# @title 7. M1: detector de placas + OCR (mismo modelo del proyecto anterior) y precisión del OCR
import re, urllib.request
import onnxruntime as ort
from fast_plate_ocr import LicensePlateRecognizer

URL_DETECTOR = "https://github.com/ankandrew/open-image-models/releases/download/assets/yolo-v9-t-384-license-plates-end2end.onnx"
DETECTOR_ONNX = MODELOS / "placas_yolo_v9_t_384.onnx"
if not DETECTOR_ONNX.exists():
    urllib.request.urlretrieve(URL_DETECTOR, DETECTOR_ONNX)

proveedores = [p for p in ("CUDAExecutionProvider", "CPUExecutionProvider") if p in ort.get_available_providers()]
detector_placas = ort.InferenceSession(str(DETECTOR_ONNX), providers=proveedores)
ocr = LicensePlateRecognizer("cct-xs-v2-global-model")
LADO_PLACAS = 384
FORMATO_BOLIVIA = re.compile(r"^\d{3,4}[A-Z]{3}$")

def detectar_placas(img, umbral=0.25):
    """Cajas (x1, y1, x2, y2, confianza) de placas en la imagen BGR."""
    alto, ancho = img.shape[:2]
    escala = min(LADO_PLACAS / alto, LADO_PLACAS / ancho)
    nw, nh = round(ancho * escala), round(alto * escala)
    mx, my = (LADO_PLACAS - nw) / 2, (LADO_PLACAS - nh) / 2
    lienzo = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    lienzo = cv2.copyMakeBorder(lienzo, round(my - 0.1), round(my + 0.1), round(mx - 0.1), round(mx + 0.1),
                                cv2.BORDER_CONSTANT, value=(114, 114, 114))
    entrada = (lienzo[:, :, ::-1].transpose(2, 0, 1) / 255.0).astype(np.float32)[None]
    salida = detector_placas.run(None, {detector_placas.get_inputs()[0].name: entrada})[0]
    cajas = []
    for _, x1, y1, x2, y2, _, conf in salida:
        if conf >= umbral:
            x1, x2 = [(v - mx) / escala for v in (x1, x2)]
            y1, y2 = [(v - my) / escala for v in (y1, y2)]
            cajas.append((int(max(0, x1)), int(max(0, y1)), int(min(ancho, x2)), int(min(alto, y2)), float(conf)))
    return cajas

def leer_texto(img, caja):
    x1, y1, x2, y2, _ = caja
    recorte = img[y1:y2, x1:x2]
    if recorte.size == 0:
        return None, 0.0
    p = ocr.run_one(cv2.cvtColor(recorte, cv2.COLOR_BGR2RGB), return_confidence=True)
    texto = re.sub(r"[^A-Z0-9]", "", (p.plate or "").upper())
    conf = float(np.mean(p.char_probs)) if p.char_probs is not None else 0.0
    return (texto or None), conf

# Precisión del OCR: pon en Drive/peaje/placas_validacion/ fotos llamadas con su placa real (ej. 1234ABC.jpg).
def precision_ocr(carpeta=CARPETA / "placas_validacion"):
    fotos = [p for p in carpeta.glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")]
    if not fotos:
        return None
    exactas, caracteres, total_car = 0, 0, 0
    for foto in fotos:
        real = re.sub(r"[^A-Z0-9]", "", foto.stem.upper().split("_")[0])
        img = cv2.imread(str(foto))
        cajas = detectar_placas(img)
        leida = leer_texto(img, max(cajas, key=lambda c: c[4]))[0] if cajas else ""
        leida = leida or ""
        exactas += leida == real
        caracteres += sum(a == b for a, b in zip(leida, real))
        total_car += max(len(real), 1)
    return {"imagenes": len(fotos), "placa_exacta": round(exactas / len(fotos), 3),
            "caracteres_correctos": round(caracteres / total_car, 3)}

METRICAS["M1_OCR"] = precision_ocr()
print("Precisión OCR:", METRICAS["M1_OCR"] or "sin fotos en placas_validacion/ (agrega algunas para medirla)")
_ = _ = _ = (INFORME / "metricas.json").write_text(json.dumps(METRICAS, indent=2, ensure_ascii=False))

In [ ]:
# @title 8. Cargar los modelos para la inferencia
import unicodedata

def sin_tildes(t):
    return "".join(c for c in unicodedata.normalize("NFD", str(t).lower()) if unicodedata.category(c) != "Mn")

SINONIMOS = {"car": "auto", "auto": "auto", "automovil": "auto", "motorcycle": "moto", "moto": "moto",
             "motocicleta": "moto", "bus": "bus", "truck": "camion", "camion": "camion", "minibus": "minibus",
             "van": "minibus", "micro": "minibus"}
CATEGORIAS = ("moto", "auto", "minibus", "camion", "bus")
DISPOSITIVO = 0 if torch.cuda.is_available() else "cpu"

coco = YOLO("yolo26n-seg.pt")                 # personas (pasajeros) y respaldo de M2
M2_PROPIO = M2_PESOS.exists()
m2 = YOLO(str(M2_PESOS)) if M2_PROPIO else coco
M3_PROPIO = M3_PESOS.exists()
m3 = YOLO(str(M3_PESOS)) if M3_PROPIO else YOLO("yolov8s-oiv7.pt")

# Clases de Open Images (respaldo de M3): ruedas y furgonetas (para minibús).
OI_RUEDAS, OI_VAN = set(), set()
if not M3_PROPIO:
    for i, n in m3.names.items():
        if n in ("Wheel", "Tire"): OI_RUEDAS.add(i)
        if n == "Van": OI_VAN.add(i)

MODELOS_USADOS = {
    "M1": "YOLOv9-t 384 (placas) + CCT-XS v2 (OCR)",
    "M2": "propio (Roboflow)" if M2_PROPIO else "YOLO26n-seg preentrenado (COCO)",
    "M3": "propio YOLO-pose (Roboflow)" if M3_PROPIO else "YOLOv8s preentrenado (Open Images V7)",
    "pasajeros": "YOLO26n-seg (COCO, clase persona)",
}
# Calentamiento: la primera inferencia en GPU siempre es lenta.
_negra = np.zeros((480, 640, 3), np.uint8)
for modelo in {id(coco): coco, id(m2): m2, id(m3): m3}.values():
    modelo.predict(_negra, device=DISPOSITIVO, verbose=False)

# ---- Agente conversacional: Qwen con Ollama (corre en la misma GPU, en otro proceso) ----
# El LLM solo redacta: los datos de las placas los consulta Laravel en su base de datos
# (function calling: Laravel ejecuta la herramienta consultar_placa y le pasa el resultado).
import shutil, time, requests

OLLAMA = "http://127.0.0.1:11434"
AGENTE_LISTO = False

def chat_agente(mensajes, herramientas=None, timeout=120):
    """Un turno del LLM. mensajes/herramientas con el formato de Ollama /api/chat (como OpenAI)."""
    datos = {"model": AGENTE_MODELO, "messages": mensajes, "stream": False, "keep_alive": -1,
             "options": {"temperature": 0.2, "num_ctx": 4096}}
    if herramientas:
        datos["tools"] = herramientas
    if AGENTE_MODELO.startswith("qwen3"):
        datos["think"] = False          # sin "razonamiento" visible: responde mucho más rápido
    r = requests.post(OLLAMA + "/api/chat", json=datos, timeout=timeout)
    r.raise_for_status()
    return r.json()["message"]

def cargar_agente():
    """Instala Ollama, lo arranca USANDO LA GPU y carga Qwen. Se puede volver a llamar sin reiniciar nada:
    detén la celda 11, ejecuta cargar_agente() en una celda nueva y vuelve a ejecutar la celda 11 (el link no cambia)."""
    global AGENTE_LISTO
    AGENTE_LISTO = False
    try:
        if shutil.which("ollama") is None:
            print("Instalando Ollama (1 a 2 minutos)...")
            subprocess.run("apt-get update -qq > /dev/null 2>&1; apt-get install -y -qq zstd pciutils lshw > /dev/null 2>&1; "
                           "curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_instalacion.log 2>&1", shell=True)
            if shutil.which("ollama") is None:
                raise RuntimeError("Ollama no se instaló (ver /tmp/ollama_instalacion.log)")
        # Reinicia Ollama por si quedó uno corriendo sin GPU.
        subprocess.run(["pkill", "-f", "ollama serve"], stderr=subprocess.DEVNULL)
        time.sleep(2)
        # En Colab las librerías de NVIDIA están en /usr/lib64-nvidia: sin esto Ollama usa solo la CPU (muy lento).
        entorno = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_KEEP_ALIVE="-1",
                       LD_LIBRARY_PATH="/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", ""))
        subprocess.Popen(["ollama", "serve"], env=entorno, stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
        for _ in range(60):
            try:
                if requests.get(OLLAMA + "/api/version", timeout=2).ok:
                    break
            except requests.RequestException:
                time.sleep(1)
        else:
            raise RuntimeError("Ollama no arrancó (ver /tmp/ollama.log)")
        print(f"Descargando {AGENTE_MODELO}...")
        r = subprocess.run(["ollama", "pull", AGENTE_MODELO], env=entorno, capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("no se pudo descargar el modelo: " + (r.stderr or r.stdout)[-300:])
        print("Cargando el modelo en la GPU (la primera vez tarda hasta 1 minuto)...")
        t = time.time()
        respuesta = chat_agente([{"role": "user", "content": "Responde solo con la palabra: listo"}], timeout=600)
        # ¿Quedó en la GPU? (size_vram > 0)
        cargados = requests.get(OLLAMA + "/api/ps", timeout=5).json().get("models", [])
        en_gpu = any(m.get("size_vram", 0) > 0 for m in cargados)
        AGENTE_LISTO = True
        print(f"Agente listo: {AGENTE_MODELO} en {'GPU' if en_gpu else 'CPU (LENTO: revisa que el entorno sea GPU T4)'} · "
              f"prueba «{respuesta.get('content', '').strip()[:30]}» en {time.time() - t:.1f} s")
    except Exception as e:
        print("AVISO: el agente no se pudo cargar; la inferencia de imágenes funciona igual.", e)
        print("Últimas líneas de /tmp/ollama.log:")
        try:
            print("".join(open("/tmp/ollama.log").readlines()[-15:]))
        except OSError:
            pass
        print("Para reintentar: detén la celda 11, ejecuta cargar_agente() en una celda nueva y luego la celda 11.")
    MODELOS_USADOS["agente"] = AGENTE_MODELO + " (Ollama)" if AGENTE_LISTO else None
    return AGENTE_LISTO

if USAR_AGENTE:
    cargar_agente()
print(json.dumps(MODELOS_USADOS, indent=2, ensure_ascii=False))

In [ ]:
# @title 9. analizar(): TODOS los vehículos, ruedas, personas y placas (mismo JSON que espera Laravel)
import base64, time
from contextlib import contextmanager

LADO_MAXIMO = 1280
EJES_POR_DEFECTO = {"moto": 2, "auto": 2, "minibus": 2, "camion": 2, "bus": 2}
# Tamaños de entrada: más grandes encuentran vehículos, personas y ruedas pequeñas o lejanas.
# En la GPU apenas suman milisegundos (a 640 px se perdían casi todas las ruedas).
TAMANO_M2 = 960
TAMANO_M3 = 1280
CONF_VEHICULOS = 0.25
CONF_RUEDAS = 0.15
MAX_VEHICULOS_PLACA = 12   # vehículos (los más grandes) donde además se busca la placa en un recorte

def iou(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

def dentro(caja, contenedor):
    """Fracción del área de `caja` que cae dentro de `contenedor`."""
    x1, y1 = max(caja[0], contenedor[0]), max(caja[1], contenedor[1])
    x2, y2 = min(caja[2], contenedor[2]), min(caja[3], contenedor[3])
    area = max(1e-6, (caja[2] - caja[0]) * (caja[3] - caja[1]))
    return max(0, x2 - x1) * max(0, y2 - y1) / area

def area(c):
    return max(0.0, c[2] - c[0]) * max(0.0, c[3] - c[1])

def detecciones(resultado):
    """Lista de {caja, conf, clase, nombre, poligono} de un resultado de Ultralytics."""
    if resultado.boxes is None or len(resultado.boxes) == 0:
        return []
    poligonos = resultado.masks.xy if resultado.masks is not None else [None] * len(resultado.boxes)
    salida = []
    for caja, conf, clase, pol in zip(resultado.boxes.xyxy.cpu().numpy(), resultado.boxes.conf.cpu().numpy(),
                                      resultado.boxes.cls.cpu().numpy().astype(int), poligonos):
        salida.append({"caja": [float(v) for v in caja], "conf": float(conf), "clase": int(clase),
                       "nombre": resultado.names[int(clase)], "poligono": pol})
    return salida

def dueno(caja, vehiculos, minimo=0.5):
    """Índice del vehículo que contiene `caja` (el más chico si hay varios: el que está adelante)."""
    candidatos = [(area(v["caja"]), i) for i, v in enumerate(vehiculos) if dentro(caja, v["caja"]) >= minimo]
    return min(candidatos)[1] if candidatos else None

def todas_las_placas(img, vehiculos):
    """Placas en la imagen completa y en el recorte de cada vehículo grande (encuentra las lejanas)."""
    cajas = list(detectar_placas(img))
    alto, ancho = img.shape[:2]
    for v in sorted(vehiculos, key=lambda v: -area(v["caja"]))[:MAX_VEHICULOS_PLACA]:
        x1, y1, x2, y2 = [int(round(c)) for c in v["caja"]]
        if x2 - x1 < 60 or y2 - y1 < 40:
            continue
        for (a, b, c, d, conf) in detectar_placas(img[max(0, y1):min(alto, y2), max(0, x1):min(ancho, x2)]):
            cajas.append((a + max(0, x1), b + max(0, y1), c + max(0, x1), d + max(0, y1), conf))
    unicas = []  # sin duplicados: se queda la de mayor confianza
    for c in sorted(cajas, key=lambda c: -c[4]):
        if all(iou(c[:4], u[:4]) < 0.3 for u in unicas):
            unicas.append(c)
    placas = []
    for c in unicas:
        texto, conf = leer_texto(img, c)
        placas.append({"caja": [int(v) for v in c[:4]], "deteccion": float(c[4]), "texto": texto, "confianza": conf})
    return placas

def ejes_de(vehiculo, ruedas):
    """Ruedas de un costado del vehículo (sin las del costado lejano ni duplicados) -> ejes."""
    x1, y1, x2, y2 = vehiculo["caja"]
    if not ruedas:
        return EJES_POR_DEFECTO.get(vehiculo["categoria"], 2), True, []
    alto_max = max(r["caja"][3] - r["caja"][1] for r in ruedas)
    cerca = sorted([r for r in ruedas if (r["caja"][3] - r["caja"][1]) >= 0.6 * alto_max], key=lambda r: r["punto"][0])
    unicas = []
    for r in cerca:
        if not unicas or abs(r["punto"][0] - unicas[-1]["punto"][0]) > (x2 - x1) * 0.06:
            unicas.append(r)
    if len(unicas) >= 2:
        return len(unicas), False, unicas
    return EJES_POR_DEFECTO.get(vehiculo["categoria"], 2), True, unicas

def analizar(img):
    inicio = time.perf_counter()
    tiempos = {}

    @contextmanager
    def medir(nombre):
        t = time.perf_counter()
        yield
        tiempos[nombre] = round((time.perf_counter() - t) * 1000)

    alto, ancho = img.shape[:2]
    if max(alto, ancho) > LADO_MAXIMO:
        escala = LADO_MAXIMO / max(alto, ancho)
        img = cv2.resize(img, (round(ancho * escala), round(alto * escala)), interpolation=cv2.INTER_AREA)
        alto, ancho = img.shape[:2]

    # M2: TODOS los vehículos (segmentación) y TODAS las personas.
    with medir("m2_segmentacion"):
        d2 = detecciones(m2.predict(img, imgsz=TAMANO_M2, conf=CONF_VEHICULOS, device=DISPOSITIVO, verbose=False)[0])
        vehiculos = [d for d in d2 if SINONIMOS.get(sin_tildes(d["nombre"])) in CATEGORIAS]
        if M2_PROPIO:
            personas = detecciones(coco.predict(img, imgsz=TAMANO_M2, conf=CONF_VEHICULOS, classes=[0],
                                                device=DISPOSITIVO, verbose=False)[0])
        else:
            personas = [d for d in d2 if d["clase"] == 0]
    for v in vehiculos:
        v["categoria"] = SINONIMOS[sin_tildes(v["nombre"])]

    # M3: TODAS las ruedas (keypoint en el centro de cada una).
    with medir("m3_ruedas"):
        r3 = m3.predict(img, imgsz=TAMANO_M3, conf=CONF_RUEDAS, device=DISPOSITIVO, verbose=False)[0]
        d3 = detecciones(r3)
        if M3_PROPIO:
            puntos = r3.keypoints.xy.cpu().numpy() if r3.keypoints is not None else []
            ruedas = [{"caja": d["caja"], "punto": [float(p[0][0]), float(p[0][1])]} for d, p in zip(d3, puntos)]
            vans = []
        else:
            ruedas = [{"caja": d["caja"], "punto": [(d["caja"][0] + d["caja"][2]) / 2, (d["caja"][1] + d["caja"][3]) / 2]}
                      for d in d3 if d["clase"] in OI_RUEDAS]
            vans = [d for d in d3 if d["clase"] in OI_VAN and d["conf"] >= CONF_VEHICULOS]

    # Minibús: COCO no lo tiene; lo marca una "Van" de Open Images sobre el mismo vehículo.
    if not M2_PROPIO:
        for v in vehiculos:
            if v["categoria"] in ("auto", "bus", "camion") and any(iou(v["caja"], van["caja"]) > 0.5 for van in vans):
                v["categoria"] = "minibus"

    # Cada rueda, persona y placa se asigna al vehículo que la contiene.
    for v in vehiculos:
        v["ruedas"], v["personas"], v["placa"] = [], [], None
    for r in ruedas:
        caja_punto = [r["punto"][0] - 1, r["punto"][1] - 1, r["punto"][0] + 1, r["punto"][1] + 1]
        i = dueno(caja_punto, vehiculos)
        if i is not None and r["punto"][1] >= vehiculos[i]["caja"][1] + 0.4 * (vehiculos[i]["caja"][3] - vehiculos[i]["caja"][1]):
            vehiculos[i]["ruedas"].append(r)
    for p in personas:
        i = dueno(p["caja"], vehiculos)
        if i is not None:
            vehiculos[i]["personas"].append(p)

    # M1: TODAS las placas + OCR.
    with medir("m1_placa_ocr"):
        placas = todas_las_placas(img, vehiculos)
    for pl in placas:
        i = dueno(pl["caja"], vehiculos)
        if i is not None:
            actual = vehiculos[i]["placa"]
            if actual is None or pl["deteccion"] > actual["deteccion"]:
                vehiculos[i]["placa"] = pl

    for v in vehiculos:
        v["ejes"], v["ejes_estimados"], v["ruedas_eje"] = ejes_de(v, v["ruedas"])

    # Vehículo principal (los campos de arriba del JSON, para el paso de la caseta):
    # el más grande que tenga placa leída; si ninguno tiene, el más grande.
    con_placa = [v for v in vehiculos if v["placa"] and v["placa"]["texto"]]
    principal = max(con_placa or vehiculos, key=lambda v: area(v["caja"]), default=None)
    placa_principal = principal["placa"] if principal else None
    if placa_principal is None and placas:  # placa sola, sin vehículo visible
        placa_principal = max(placas, key=lambda p: p["deteccion"])

    resumen = {
        "placa": placa_principal["texto"] if placa_principal else None,
        "confianza": round(placa_principal["confianza"], 4) if placa_principal and placa_principal["texto"] else None,
        "categoria": principal["categoria"] if principal else None,
        "ejes": principal["ejes"] if principal else None,
        "pasajeros": len(principal["personas"]) if principal else None,
    }

    lista = []
    for v in sorted(vehiculos, key=lambda v: -area(v["caja"])):
        pl = v["placa"]
        lista.append({
            "categoria": v["categoria"], "confianza": round(v["conf"], 3), "caja": [round(c) for c in v["caja"]],
            "ejes": v["ejes"], "ejes_estimados": v["ejes_estimados"], "pasajeros": len(v["personas"]),
            "placa": pl["texto"] if pl else None,
            "confianza_placa": round(pl["confianza"], 4) if pl and pl["texto"] else None,
            "caja_placa": pl["caja"] if pl else None,
            "llantas": [[round(r["punto"][0]), round(r["punto"][1])] for r in v["ruedas_eje"]],
            "principal": v is principal,
        })

    with medir("evidencia"):
        evidencia = dibujar_evidencia(img, vehiculos, personas, ruedas, placas, resumen)

    return resumen | {
        "latencia_ms": round((time.perf_counter() - inicio) * 1000),
        "evidencia": evidencia,
        "vehiculos": lista,
        "detalle": {
            "vehiculo": {"clase": principal["categoria"], "confianza": round(principal["conf"], 3),
                         "caja": [round(v) for v in principal["caja"]]} if principal else None,
            "placa": {"caja": placa_principal["caja"], "confianza_deteccion": round(placa_principal["deteccion"], 3)} if placa_principal else None,
            "llantas": [[round(r["punto"][0]), round(r["punto"][1])] for r in (principal["ruedas_eje"] if principal else [])],
            "ejes_estimados": principal["ejes_estimados"] if principal else True,
            "vehiculos_detectados": len(vehiculos),
            "personas_detectadas": len(personas),
            "llantas_detectadas": len(ruedas),
            "placas_detectadas": len(placas),
            "placas_leidas": [p["texto"] for p in placas if p["texto"]],
            "imagen": [ancho, alto],
            "modelos": MODELOS_USADOS,
            "dispositivo": DIAGNOSTICO["gpu"] or "cpu",
            "tiempos_ms": tiempos,
        },
    }

AZUL, VERDE, GRIS, ROSA, OSCURO, AMARILLO = (233, 165, 14), (129, 185, 16), (139, 116, 100), (153, 72, 236), (42, 23, 15), (11, 158, 245)

def etiqueta(lienzo, texto, x, y, fondo, escala=0.55):
    (tw, th), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, escala, 1)
    y = max(th + 6, y)
    cv2.rectangle(lienzo, (x, y - th - 6), (x + tw + 6, y), fondo, -1)
    cv2.putText(lienzo, texto, (x + 3, y - 4), cv2.FONT_HERSHEY_SIMPLEX, escala, (255, 255, 255), 1, cv2.LINE_AA)

def dibujar_evidencia(img, vehiculos, personas, ruedas, placas, resumen):
    """Todos los vehículos (máscara), todas las ruedas, todas las placas. Personas cubiertas (solo conteo)."""
    lienzo = img.copy()
    grosor = max(2, round(lienzo.shape[1] / 500))
    capa = lienzo.copy()
    for v in vehiculos:
        if v["poligono"] is not None and len(v["poligono"]) > 2:
            cv2.fillPoly(capa, [v["poligono"].astype(np.int32)], AMARILLO if v.get("placa") else AZUL)
    lienzo = cv2.addWeighted(capa, 0.3, lienzo, 0.7, 0)
    for p in personas:  # privacidad: silueta en color sólido, sin rostro
        if p["poligono"] is not None and len(p["poligono"]) > 2:
            cv2.fillPoly(lienzo, [p["poligono"].astype(np.int32)], GRIS)
        else:
            x1, y1, x2, y2 = map(round, p["caja"])
            cv2.rectangle(lienzo, (x1, y1), (x2, y2), GRIS, -1)
    for n, v in enumerate(sorted(vehiculos, key=lambda v: -area(v["caja"])), 1):
        x1, y1, x2, y2 = map(round, v["caja"])
        cv2.rectangle(lienzo, (x1, y1), (x2, y2), AZUL, grosor)
        texto = f"{n} {v['categoria'].upper()} {v['ejes']}ej" + (f" {len(v['personas'])}p" if v["personas"] else "")
        etiqueta(lienzo, texto, x1, y1, AZUL)
    for r in ruedas:  # todas las ruedas detectadas (keypoint en el centro)
        cx, cy = map(round, r["punto"])
        radio = max(4, round((r["caja"][3] - r["caja"][1]) * 0.12))
        cv2.circle(lienzo, (cx, cy), radio + 2, (255, 255, 255), -1)
        cv2.circle(lienzo, (cx, cy), radio, ROSA, -1)
    for pl in placas:
        x1, y1, x2, y2 = pl["caja"]
        cv2.rectangle(lienzo, (x1, y1), (x2, y2), VERDE, grosor)
        etiqueta(lienzo, f"{pl['texto'] or '?'} {round(pl['confianza'] * 100)}%", x1, y2 + 18, VERDE)
    encabezado = (f"{len(vehiculos)} vehiculo(s) | {len(ruedas)} rueda(s) | {len(personas)} persona(s) | "
                  f"{len([p for p in placas if p['texto']])} placa(s)  >  principal: "
                  + (resumen["categoria"].upper() if resumen["categoria"] else "-") + " " + (resumen["placa"] or "---"))
    etiqueta(lienzo, encabezado, 0, 26, OSCURO, 0.6)
    # La evidencia viaja de vuelta por el túnel (lento): se achica a 960 px. Los datos no cambian.
    alto_e, ancho_e = lienzo.shape[:2]
    if max(alto_e, ancho_e) > 960:
        f = 960 / max(alto_e, ancho_e)
        lienzo = cv2.resize(lienzo, (round(ancho_e * f), round(alto_e * f)), interpolation=cv2.INTER_AREA)
    return base64.b64encode(cv2.imencode(".jpg", lienzo, [cv2.IMWRITE_JPEG_QUALITY, 70])[1]).decode()  # liviana: el túnel es lento

# Prueba: bus en Tiwanaku, Bolivia, placa 4027UGL (Wikimedia Commons, CC BY 2.0, autor Sasha India).
# Si Wikimedia no responde, se usa la foto de ejemplo de Ultralytics (un bus con personas).
FOTOS_PRUEBA = [
    "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3c/Bus_to_Tiwanaku_%28Bolivia%29_%2836797129561%29.jpg/1280px-Bus_to_Tiwanaku_%28Bolivia%29_%2836797129561%29.jpg",
    "https://ultralytics.com/images/bus.jpg",
]

def bajar_imagen(url):
    pedido = urllib.request.Request(url, headers={"User-Agent": "peaje-colab/1.0 (proyecto educativo)"})
    datos = np.frombuffer(urllib.request.urlopen(pedido, timeout=30).read(), np.uint8)
    return cv2.imdecode(datos, cv2.IMREAD_COLOR)

for url in FOTOS_PRUEBA:
    try:
        img_prueba = bajar_imagen(url)
    except Exception as e:
        print("No se pudo bajar la foto", url.split("/")[-1][:40], "(se prueba con la siguiente):", str(e)[:60])
        continue
    if img_prueba is None:
        continue
    prueba = analizar(img_prueba)
    print(json.dumps({k: v for k, v in prueba.items() if k != "evidencia"}, indent=2, ensure_ascii=False))
    if EN_COLAB:
        from IPython.display import Image as IImage, display
        display(IImage(data=base64.b64decode(prueba["evidencia"]), width=720))
    break

In [ ]:
# @title 10. Endpoint de inferencia (FastAPI) + túnel público de Cloudflare
import secrets, threading, shutil, requests, uvicorn
from fastapi import FastAPI, File, UploadFile, Header, HTTPException, Body

# Si la celda se vuelve a ejecutar, se mantiene el mismo token (y abajo, el mismo servidor y link).
TOKEN = INFERENCE_TOKEN or globals().get("TOKEN") or secrets.token_urlsafe(24)
BLOQUEO = threading.Lock()   # una inferencia a la vez en la GPU; las peticiones esperan su turno
ESTADISTICAS = {"peticiones": 0, "latencias": []}
MAXIMO_BYTES = 12 * 1024 * 1024

app = FastAPI(title="Inferencia del peaje", version="1.1",
              description="M1 placa + OCR, M2 segmentación (categoría), M3 ruedas (ejes), pasajeros y agente (Qwen).")

@app.get("/")
def inicio():
    return {"servicio": "inferencia del peaje", "endpoints": ["/health", "/inferir", "/agente/chat"]}

@app.get("/health")
def health():
    return {"estado": "ok", "modelos": ["M1", "M2", "M3"], "detalle": MODELOS_USADOS,
            "agente": AGENTE_MODELO if AGENTE_LISTO else None,
            "gpu": DIAGNOSTICO["gpu"], "peticiones": ESTADISTICAS["peticiones"]}

@app.post("/agente/chat")
def agente(cuerpo: dict = Body(...), x_api_key: str = Header(default="")):
    """Un turno del LLM. Laravel manda {mensajes, herramientas} y ejecuta él mismo las herramientas."""
    if not secrets.compare_digest(x_api_key, TOKEN):
        raise HTTPException(status_code=401, detail="Token inválido.")
    if not AGENTE_LISTO:
        raise HTTPException(status_code=503, detail="El agente no está cargado (celda 8).")
    mensajes = cuerpo.get("mensajes")
    if not isinstance(mensajes, list) or not mensajes:
        raise HTTPException(status_code=422, detail="Faltan los mensajes.")
    inicio_t = time.time()
    try:
        mensaje = chat_agente(mensajes, cuerpo.get("herramientas") or None)
    except Exception as e:
        raise HTTPException(status_code=502, detail=f"Ollama falló: {e}")
    return {"mensaje": mensaje, "modelo": AGENTE_MODELO, "latencia_ms": round((time.time() - inicio_t) * 1000)}

@app.post("/inferir")
def inferir(imagen: UploadFile = File(...), x_api_key: str = Header(default="")):
    if not secrets.compare_digest(x_api_key, TOKEN):
        raise HTTPException(status_code=401, detail="Token inválido.")
    contenido = imagen.file.read()
    if not contenido or len(contenido) > MAXIMO_BYTES:
        raise HTTPException(status_code=413 if contenido else 400, detail="Imagen vacía o mayor a 12 MB.")
    img = cv2.imdecode(np.frombuffer(contenido, np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise HTTPException(status_code=400, detail="No se pudo leer la imagen.")
    with BLOQUEO:
        resultado = analizar(img)
    ESTADISTICAS["peticiones"] += 1
    ESTADISTICAS["latencias"] = (ESTADISTICAS["latencias"] + [resultado["latencia_ms"]])[-200:]
    return resultado

def _api_responde():
    try:
        return requests.get(f"http://127.0.0.1:{PUERTO}/health", timeout=2).ok
    except requests.RequestException:
        return False

if _api_responde():
    # Celda ejecutada otra vez: el servidor anterior sigue corriendo (lee estas mismas variables).
    print("El servidor ya estaba corriendo: se reutiliza.")
else:
    servidor = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PUERTO, log_level="warning"))
    threading.Thread(target=servidor.run, daemon=True).start()
for _ in range(60):
    try:
        if requests.get(f"http://127.0.0.1:{PUERTO}/health", timeout=2).ok:
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("La API no arrancó.")

TUNEL_VIVO = "tunel" in globals() and tunel.poll() is None and str(globals().get("URL_PUBLICA", "")).startswith("https://")
if not TUNEL_VIVO:
    URL_PUBLICA = f"http://127.0.0.1:{PUERTO}"
if USAR_TUNEL and TUNEL_VIVO:
    print("El túnel ya estaba abierto: el link no cambia.")
elif USAR_TUNEL:
    cloudflared = shutil.which("cloudflared") or "/usr/local/bin/cloudflared"
    if not Path(cloudflared).exists():
        urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cloudflared)
        os.chmod(cloudflared, 0o755)
    registro = open("/tmp/cloudflared.log", "w")
    tunel = subprocess.Popen([cloudflared, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PUERTO}"],
                             stdout=registro, stderr=subprocess.STDOUT)
    for _ in range(90):
        hallado = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", Path("/tmp/cloudflared.log").read_text())
        if hallado:
            URL_PUBLICA = hallado.group(0)
            break
        time.sleep(1)
    else:
        raise RuntimeError("Cloudflare no dio una URL. Vuelve a ejecutar esta celda.")
    for _ in range(30):  # el DNS del túnel tarda unos segundos
        try:
            if requests.get(URL_PUBLICA + "/health", timeout=5).ok:
                break
        except requests.RequestException:
            pass
        time.sleep(2)

print("=" * 64)
print("En la web: Estado de la IA → Conexión con Colab → pega el link y el token → Guardar y probar")
print("=" * 64)
print("Link:  " + URL_PUBLICA)
print("Token: " + TOKEN)
print("Agente:", AGENTE_MODELO if AGENTE_LISTO else "no cargado")
print("=" * 64)
print("(Alternativa: en el .env de Laravel INFERENCE_DRIVER=http, INFERENCE_URL=<link>, INFERENCE_TOKEN=<token>)")
_ = (INFORME / "endpoint.txt").write_text(f"{datetime.datetime.now()}  {URL_PUBLICA}\n")

In [ ]:
# @title 11. Mantener la sesión activa y ver la latencia (detén esta celda para terminar)
# Colab cierra la sesión si no hay actividad. Mientras esta celda corre, el endpoint sigue disponible.
import statistics
try:
    while True:
        time.sleep(60)
        lat = ESTADISTICAS["latencias"]
        if lat:
            print(f"{time.strftime('%H:%M:%S')}  peticiones={ESTADISTICAS['peticiones']}  "
                  f"latencia media={statistics.mean(lat):.0f} ms  p95={sorted(lat)[int(len(lat) * 0.95) - 1]:.0f} ms")
        else:
            print(f"{time.strftime('%H:%M:%S')}  esperando peticiones en {URL_PUBLICA}")
except KeyboardInterrupt:
    print("Detenido. El endpoint deja de funcionar al cerrar la sesión de Colab.")